In [1]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent))
from dotenv import load_dotenv
load_dotenv()

True

In [ ]:
import pandas as pd
import ipywidgets as widgets
from IPython.display import display, clear_output
from scripts.search_eval import run

In [3]:
query_input = widgets.Text(
    placeholder='e.g. αvβ6 integrin IPF TGF-β',
    layout=widgets.Layout(width='500px')
)
max_results_input = widgets.BoundedIntText(value=25, min=1, max=200, description='Max papers:')
run_button = widgets.Button(description='Search & Evaluate', button_style='primary')
progress_output = widgets.Output()
results_output = widgets.Output()
rationale_output = widgets.Output()

TABLE_COLS = ["pmid", "title", "confidence", "overall", "model", "pathway", "study_design", "rationale_summary"]
COL_LABELS = ["PMID", "Title", "Conf", "Ovrl", "Mdl", "Pth", "SDS", "Rationale"]

def on_run(b):
    query = query_input.value.strip()
    if not query:
        return
    run_button.disabled = True
    run_button.description = 'Running…'

    results_output.clear_output()
    rationale_output.clear_output()

    with progress_output:
        clear_output()
        rows = run(query, max_results_input.value)

    with results_output:
        if not rows:
            print('No papers found.')
        else:
            df = pd.DataFrame(rows)[TABLE_COLS]
            df.columns = COL_LABELS
            df['Conf'] = df['Conf'].map('{:.3f}'.format)
            df['Ovrl'] = df['Ovrl'].map('{:.3f}'.format)
            df['Mdl']  = df['Mdl'].map('{:.3f}'.format)
            df['Pth']  = df['Pth'].map('{:.3f}'.format)
            df['SDS']  = df['SDS'].map('{:.3f}'.format)
            display(df.style.set_properties(**{'text-align': 'left'}).hide(axis='index'))

    with rationale_output:
        if rows:
            panels = []
            for r in rows:
                src = ' [ChromaDB]' if r.get('from_chroma') else ''
                lines = []
                if r.get('abstract'):
                    lines.append(f"Abstract: {r['abstract']}\n")
                lines.append(f"Warnings: {r['warnings']}")
                lines.append(f"Detected models: {r['detected_models']}")
                lines.append(f"Pathways: {r['detected_pathways']}")
                lines.append(f"Contested flags: {r['contested_flags']}")
                lines += r['rationale_lines']
                out = widgets.Output()
                with out:
                    print('\n'.join(lines))
                panels.append(out)
            acc = widgets.Accordion(children=panels)
            for i, r in enumerate(rows):
                label = f"[{r['confidence']:.3f}] {r['pmid']} — {r['title'][:70]}{src}"
                acc.set_title(i, label)
            acc.selected_index = None
            display(acc)

    run_button.disabled = False
    run_button.description = 'Search & Evaluate'

run_button.on_click(on_run)

In [4]:
display(widgets.VBox([
    widgets.HTML('<h3>FibrosisLit Evidence Search</h3>'),
    widgets.HBox([query_input, max_results_input, run_button]),
    widgets.HTML('<b>Pipeline output</b>'),
    progress_output,
    widgets.HTML('<b>Results</b>'),
    results_output,
    widgets.HTML('<b>Full Rationale</b>'),
    rationale_output,
]))